In [ ]:
from dotenv import load_dotenv
import os
load_dotenv()

True

In [29]:
Gemini_API_Key = os.getenv("Gemini_API_Key")

In [30]:
if not Gemini_API_Key:
    raise ValueError("Gemini_API_Key is missing in your .env")
os.environ["Gemini_API_Key"]=Gemini_API_Key

In [22]:
from langchain_google_genai import GoogleGenerativeAI
from langchain_core.messages import HumanMessage

In [23]:
chat_llm=GoogleGenerativeAI(model='gemini-2.5-flash')

In [28]:

response = chat_llm.invoke([HumanMessage(content="Hello")])
print(response)
# chat_llm.invoke("Hello, How are you?")

Hello! How can I help you today?


In [31]:
from typing_extensions import TypedDict, Annotated
import operator


In [32]:
from langchain_core.messages import AnyMessage, HumanMessage, AIMessage


In [33]:
class Graphstate(TypedDict):
    messages:Annotated[list[AnyMessage],operator.add]

In [34]:
def llm_call(state:operator)->dict:
    """Call the LLM using conversation message and append AI response"""
    response=chat_llm.invoke(state["messages"]) #AI Message
    return {
        "messages":[response]
    }

In [35]:
def token_counter(state:Graphstate)->dict:
    """Count tokens (simple word count) in the last AI message"""
    last_msg=state["messages"][-1]
    text = last_msg.content
    token_number = len(text.split())
    summary = f"Total token number in the generated answer (word count) is {token_number}"
    return {
        "messages":[AIMessage(content=summary)]
    }

In [36]:
from langgraph.graph import StateGraph

In [37]:
builder = StateGraph(Graphstate)

In [38]:
builder.add_node("llm_call",llm_call)
builder.add_node("token_counter",token_counter)

In [39]:
builder.set_entry_point("llm_call")
builder.add_edge("llm_call","token_counter")
builder.set_finish_point("token_counter")

In [41]:
app = builder.compile()

In [42]:
app.get_graph()

Graph(nodes={'__start__': Node(id='__start__', name='__start__', data=RunnableCallable(tags=None, recurse=True, explode_args=False, func_accepts={}), metadata=None), 'llm_call': Node(id='llm_call', name='llm_call', data=llm_call(tags=None, recurse=True, explode_args=False, func_accepts={}), metadata=None), 'token_counter': Node(id='token_counter', name='token_counter', data=token_counter(tags=None, recurse=True, explode_args=False, func_accepts={}), metadata=None), '__end__': Node(id='__end__', name='__end__', data=None, metadata=None)}, edges=[Edge(source='__start__', target='llm_call', data=None, conditional=False), Edge(source='llm_call', target='token_counter', data=None, conditional=False), Edge(source='token_counter', target='__end__', data=None, conditional=False)])